In [1]:
import os

os.environ['HF_HOME'] = '/ocean/projects/cis250042p/sjain13'

In [32]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# 1. Load the model and tokenizer
model_name = "Qwen/Qwen3-4B-Instruct-2507"


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=None, # No quantization for now
    device_map="auto",  # Automatically uses available GPUs
    trust_remote_code=True,
    output_hidden_states=True,
    local_files_only=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# Set the padding token to be the same as the end-of-sequence token
tokenizer.pad_token = tokenizer.eos_token

# 2. Configure LoRA (PEFT)
# LoRA is a technique to drastically reduce the number of trainable parameters.
lora_config = LoraConfig(
    r=16,  # The dimension of the low-rank matrices
    lora_alpha=32,  # The scaling factor for the low-rank matrices
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Apply LoRA to attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Prepare model for k-bit training and apply PEFT
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [33]:
# 3. Load and prepare the dataset
dataset_name = "training_POC/dummy_train.json"
dataset = load_dataset('json', data_files=dataset_name, split="train")

In [34]:
training_args = TrainingArguments(
    output_dir="./sft-qwen3-4b-results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit", # Memory-efficient optimizer
    logging_steps=5,
    learning_rate=2e-4,
    bf16=False, # Use bfloat16 for training if your GPU supports it
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    report_to="tensorboard",
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [35]:
from mlp_training.mlp import MLP

class MySFTTrainer(SFTTrainer):
    def __init__(self, *args, **kwargs):
        # First, call the parent class's constructor
        super().__init__(*args, **kwargs)
        # Now, initialize your custom attribute
        self.frozen_mlp = None
        self.load_mlp()

    def load_mlp(self):
        if self.frozen_mlp is None:
            print("DBG", "Loading MLP...")
            TOTAL_MLP_LAYERS, MLP_DIMS = 36, [2560, 1024, 512, 1]
            self.frozen_mlp = MLP(TOTAL_MLP_LAYERS+1, MLP_DIMS)
            self.frozen_mlp.load_state_dict(torch.load("mlp_training/q3_4_mlp_mode_lin_agt.pth"))
            self.frozen_mlp.eval()
            for param in self.frozen_mlp.parameters():
                param.requires_grad = False
            self.frozen_mlp.to(self.model.device)
            self.frozen_mlp.eval()
        
    def compute_loss(self, model, inputs, num_items_in_batch=None, return_outputs=False):
        if self.frozen_mlp is None:
            self.load_mlp()
        inputs["output_hidden_states"] = True
        outputs = model(**inputs)
        loss = outputs.loss
        # print("DBG", "Loss before MLP:", loss.item())
        
        labels = inputs.get("labels")
        hidden_states = getattr(outputs, "hidden_states", None)
        if hidden_states is not None:
            mask = labels != -100
            total_mlp_loss = 0
            for l in range(hidden_states[0].shape[1]):
                if any(mask[:, l]):    
                    hs = torch.stack([x[:,l,:] for x in hidden_states], dim=1)#, ).transpose(0,1)
                    mlp_loss = self.frozen_mlp(hs).squeeze(-1)
                    mlp_loss = mlp_loss * mask[:, l].float()
                    total_mlp_loss += mlp_loss.sum()

            # print("DBG", "MLP Loss:", total_mlp_loss.item(), mask.sum())
            loss += (total_mlp_loss / mask.sum())
        return (loss, outputs) if return_outputs else loss


def formatting_func(data):
    # Create the chat structure that the tokenizer's chat template expects
    chats = []
    for i in range(len(data['prompt'])): 
        chats.append([{"role": "user", "content": data["prompt"][i]}, {"role": "assistant", "content": data["output"][i]}])
    return tokenizer.apply_chat_template(chats, tokenize=False)

response_template = "<|im_start|>assistant\n"

# Pass this string to the collator
data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template, 
    tokenizer=tokenizer
)

trainer = MySFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    formatting_func=formatting_func,
    data_collator=data_collator,
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)


/jet/home/sjain13/miniconda3/envs/capstone/lib/python3.9/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/jet/home/sjain13/miniconda3/envs/capstone/lib/python3.9/site-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/8446 [00:00<?, ? examples/s]

/jet/home/sjain13/miniconda3/envs/capstone/lib/python3.9/site-packages/trl/trainer/sft_trainer.py:413: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `MySFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


DBG Loading MLP...


In [ ]:
trainer.train() # loss drops to 0 since output strings are all same in the dummy_train.json data

Step,Training Loss
5,35.849800
10,9.394400
15,0.082000
20,0.000100


KeyboardInterrupt: 